# 🎬 YT8M Genre Classifier — Stage 7: Inference Pipeline
YouTube ссылка или локальное видео → жанр.

**Важно:** используем официальный YT8M feature extractor (Inception V3 + VGGish + PCA)
чтобы фичи совпадали с тренировочным распределением.

In [1]:
# # ============================================================
# # STAGE 7.0 — Установка зависимостей
# # ============================================================
# # Запустите один раз, потом можно закомментировать
# import subprocess, sys
#
# packages = ['yt-dlp', 'opencv-python', 'tf_slim', 'soundfile']
# for pkg in packages:
#     result = subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'],
#                              capture_output=True, text=True)
#     status = '✅' if result.returncode == 0 else '❌'
#     print(f'{status} {pkg}')
#
# # VGGish from tensorflow/models audioset
# import urllib.request
# from pathlib import Path
#
# MODELS_DIR = Path.home() / 'yt8m' / 'models'
# MODELS_DIR.mkdir(parents=True, exist_ok=True)
# print(f'\n📂 Models dir: {MODELS_DIR}')


In [2]:
# # Фикс: откатываем numpy до совместимой версии
# import subprocess, sys
# subprocess.run([sys.executable, '-m', 'pip', 'install',
#                 'numpy==1.24.4', '-q'], check=True)
# print('✅ numpy==1.24.4 установлен')
# print('⚠️  ПЕРЕЗАПУСТИТЕ KERNEL (Kernel → Restart), затем продолжайте с 7.0')

In [3]:
# ============================================================
# STAGE 7.1 — Загружаем локальный feature extractor
# ============================================================
import sys
from pathlib import Path

# Берём локальную копию из корня проекта
PROJECT_DIR = Path('..').resolve()
sys.path.insert(0, str(PROJECT_DIR))

import feature_extractor as yt8m_fe
print('✅ feature_extractor импортирован из проекта')
print('   Первый запуск скачает Inception (~75MB) + PCA (~25MB)')

I0000 00:00:1774288128.588336  162059 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1774288128.624251  162059 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1774288129.435936  162059 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


✅ feature_extractor импортирован из проекта
   Первый запуск скачает Inception (~75MB) + PCA (~25MB)


In [4]:
# ============================================================
# STAGE 7.2 — Инициализация экстрактора и PyTorch модели
# ============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pickle, json
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

STAGE3_DIR = Path(Path('../data2/stage3'))
STAGE5_DIR = Path(Path('../data2/stage5'))
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

with open(STAGE3_DIR / 'config.json') as f:
    cfg = json.load(f)
GENRES = cfg['genres']

# Инициализируем YT8M feature extractor
# (первый раз скачает Inception + PCA матрицы ~100MB)
print('🔄 Инициализируем YT8M Feature Extractor (первый раз ~30 сек, скачивает модели)...')
extractor = yt8m_fe.YouTube8MFeatureExtractor()
print('✅ Extractor готов')

# Загружаем скейлеры из Stage 3
with open(STAGE3_DIR / 'scalers.pkl', 'rb') as f:
    scalers = pickle.load(f)
scaler_v = scalers['visual']
scaler_a = scalers['audio']
print('✅ Scalers загружены')

# Определяем модель (копия из Stage 4-5)
class ResidualBlock(nn.Module):
    def __init__(self, dim, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim), nn.BatchNorm1d(dim), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(dim, dim), nn.BatchNorm1d(dim),
        )
        self.act = nn.GELU()
    def forward(self, x): return self.act(x + self.net(x))

class VisualBranch(nn.Module):
    def __init__(self, in_dim=1024, embed_dim=256, dropout=0.3):
        super().__init__()
        self.proj = nn.Sequential(nn.Linear(in_dim, embed_dim), nn.BatchNorm1d(embed_dim), nn.GELU())
        self.res1 = ResidualBlock(embed_dim, dropout)
        self.res2 = ResidualBlock(embed_dim, dropout)
    def forward(self, x): return self.res2(self.res1(self.proj(x)))

class AudioBranch(nn.Module):
    def __init__(self, in_dim=128, embed_dim=128, dropout=0.3):
        super().__init__()
        self.proj = nn.Sequential(nn.Linear(in_dim, embed_dim), nn.BatchNorm1d(embed_dim), nn.GELU())
        self.res  = ResidualBlock(embed_dim, dropout)
    def forward(self, x): return self.res(self.proj(x))

class CrossModalAttention(nn.Module):
    def __init__(self, visual_dim=256, audio_dim=128, fusion_dim=256, n_heads=4, dropout=0.2):
        super().__init__()
        self.proj_v = nn.Linear(visual_dim, fusion_dim)
        self.proj_a = nn.Linear(audio_dim,  fusion_dim)
        self.attn   = nn.MultiheadAttention(fusion_dim, n_heads, dropout=dropout, batch_first=True)
        self.norm   = nn.LayerNorm(fusion_dim)
        self.out_dim = fusion_dim * 2
    def forward(self, v, a):
        tokens = torch.cat([self.proj_v(v).unsqueeze(1), self.proj_a(a).unsqueeze(1)], dim=1)
        out, attn_w = self.attn(tokens, tokens, tokens)
        out = self.norm(out + tokens)
        return out.reshape(out.size(0), -1), attn_w

class YT8MClassifier(nn.Module):
    def __init__(self, dim_visual=1024, dim_audio=128, n_classes=12,
                 visual_embed=256, audio_embed=128, fusion_dim=256, n_heads=4, dropout=0.35):
        super().__init__()
        self.visual_branch = VisualBranch(dim_visual, visual_embed, dropout)
        self.audio_branch  = AudioBranch(dim_audio, audio_embed, dropout)
        self.fusion = CrossModalAttention(visual_embed, audio_embed, fusion_dim, n_heads, dropout*0.5)
        fused_dim = self.fusion.out_dim
        self.classifier = nn.Sequential(
            nn.Linear(fused_dim, 256), nn.BatchNorm1d(256), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(256, 128),       nn.BatchNorm1d(128), nn.GELU(), nn.Dropout(dropout*0.5),
            nn.Linear(128, n_classes),
        )
    def forward(self, xv, xa):
        fused, attn_w = self.fusion(self.visual_branch(xv), self.audio_branch(xa))
        return self.classifier(fused), attn_w

model = YT8MClassifier(cfg['dim_visual'], cfg['dim_audio'], cfg['n_classes']).to(DEVICE)
model.load_state_dict(torch.load(STAGE5_DIR / 'best_model.pt', weights_only=True, map_location=DEVICE))
model.eval()
print('✅ YT8MClassifier загружен (best_model.pt)')


🔄 Инициализируем YT8M Feature Extractor (первый раз ~30 сек, скачивает модели)...
✅ Extractor готов
✅ Scalers загружены
✅ YT8MClassifier загружен (best_model.pt)


W0000 00:00:1774288131.792234  162059 op_def_util.cc:371] Op BatchNormWithGlobalNormalization is deprecated. It will cease to work in GraphDef version 9. Use tf.nn.batch_normalization().
W0000 00:00:1774288131.888048  162059 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
W0000 00:00:1774288131.893230  162059 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
I0000 00:00:1774288131.986778  162059 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 12459 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 5060 Ti, pci bus id: 0000:01:00.0, compute capability: 12.0a


In [5]:
# ============================================================
# STAGE 7.3 — Feature extraction pipeline
# ============================================================
import cv2
import subprocess
import tempfile
import os

GENRE_EMOJI = {
    'Gaming': '🎮', 'Music': '🎵', 'Animation': '🎨',
    'Vehicles': '🚗', 'Sports': '⚽', 'Animals': '🐾',
    'Food': '🍳', 'Dance': '💃', 'Performance': '🎭',
    'Tech': '📱', 'Beauty': '💄', 'Film': '🎬',
}

def extract_features_from_video(video_path, max_frames=60, fps_sample=1):
    """
    Извлекает YT8M-совместимые visual (1024) и audio (128) фичи из видео.
    Возвращает: (mean_rgb, mean_audio) — усреднённые по всем сэмплированным кадрам.
    """
    video_path = str(video_path)

    # === Visual: Inception V3 + PCA (официальный YT8M экстрактор) ===
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f'Не удалось открыть видео: {video_path}')

    fps_orig = cap.get(cv2.CAP_PROP_FPS) or 25
    frame_interval = max(1, int(fps_orig / fps_sample))  # берём 1 кадр в секунду

    rgb_features = []
    frame_idx = 0
    while len(rgb_features) < max_frames:
        ret, frame = cap.read()
        if not ret:
            break
        if frame_idx % frame_interval == 0:
            # BGR -> RGB, resize to 224x224 для Inception
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame_rgb = cv2.resize(frame_rgb, (224, 224))
            feat = extractor.extract_rgb_frame_features(frame_rgb)
            rgb_features.append(feat)
        frame_idx += 1
    cap.release()

    if not rgb_features:
        raise RuntimeError('Не удалось извлечь ни одного кадра')

    mean_rgb = np.mean(rgb_features, axis=0).astype(np.float32)  # (1024,)

    # === Audio: VGGish через ffmpeg + librosa ===
    # Извлекаем аудио в WAV
    with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as tmp:
        wav_path = tmp.name

    try:
        subprocess.run(
            ['ffmpeg', '-y', '-i', video_path, '-ar', '16000',
             '-ac', '1', '-f', 'wav', wav_path],
            capture_output=True, check=True
        )
        audio_feat = extractor.extract_audio_features(wav_path)
        mean_audio = np.mean(audio_feat, axis=0).astype(np.float32)  # (128,)
    except Exception as e:
        print(f'  ⚠️  Аудио не извлечено ({e}), используем нули')
        mean_audio = np.zeros(128, dtype=np.float32)
    finally:
        if os.path.exists(wav_path):
            os.remove(wav_path)

    return mean_rgb, mean_audio


def predict_genre(video_path, top_k=3, verbose=True):
    """
    Принимает путь к видео файлу, возвращает топ-k жанров с вероятностями.
    """
    if verbose:
        print(f'\n🎬 Анализируем: {Path(video_path).name}')
        print('   Извлекаем фичи...', end=' ', flush=True)

    mean_rgb, mean_audio = extract_features_from_video(video_path)

    if verbose:
        print('✅')
        print(f'   RGB shape={mean_rgb.shape}  Audio shape={mean_audio.shape}')

    # Нормализуем через Stage 3 скейлеры
    xv = scaler_v.transform(mean_rgb.reshape(1, -1)).astype(np.float32)
    xa = scaler_a.transform(mean_audio.reshape(1, -1)).astype(np.float32)

    xv_t = torch.tensor(xv).to(DEVICE)
    xa_t = torch.tensor(xa).to(DEVICE)

    with torch.no_grad():
        logits, attn_w = model(xv_t, xa_t)
        probs = F.softmax(logits, dim=-1).cpu().numpy()[0]

    top_idx  = np.argsort(probs)[::-1][:top_k]
    results  = [(GENRES[i], float(probs[i])) for i in top_idx]

    if verbose:
        print('\n   📊 Результаты:')
        for rank, (genre, prob) in enumerate(results, 1):
            bar  = '█' * int(prob * 30)
            emoj = GENRE_EMOJI.get(genre, '🎬')
            print(f'   {rank}. {emoj} {genre:<14}  {prob*100:>5.1f}%  {bar}')

    return results


print('✅ predict_genre() готова')


✅ predict_genre() готова


In [6]:
# ============================================================
# STAGE 7.4 — Скачивание видео с YouTube
# ============================================================
import subprocess
import tempfile

DOWNLOAD_DIR = Path(Path('../data2/inference'))
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

def download_youtube(url, output_dir=DOWNLOAD_DIR, max_seconds=60):
    """
    Скачивает видео с YouTube через yt-dlp.
    Берёт первые max_seconds секунд (не перекодирует, просто обрезает).
    Возвращает путь к скачанному файлу.
    """
    print(f'📥 Скачиваем: {url}')
    out_template = str(output_dir / '%(id)s.%(ext)s')
    cmd = [
        'yt-dlp',
        '-f', 'bestvideo[height<=720]+bestaudio/best[height<=720]',
        '-S', 'vcodec:h264,res:720',   # предпочитаем h264, OpenCV его точно читает
        '--merge-output-format', 'mp4',
        '-o', out_template,
        '--no-playlist',
        url
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f'❌ yt-dlp error:\n{result.stderr[-500:]}')
        return None

    # Находим скачанный файл
    mp4_files = sorted(output_dir.glob('*.mp4'), key=lambda x: x.stat().st_mtime)
    if not mp4_files:
        print('❌ Файл не найден после скачивания')
        return None

    video_path = mp4_files[-1]
    size_mb = video_path.stat().st_size / 1024**2
    print(f'✅ Скачано: {video_path.name}  ({size_mb:.1f} MB)')
    return video_path


print('✅ download_youtube() готова')
print('\n💡 Для скачивания и анализа выполните следующую ячейку')

✅ download_youtube() готова

💡 Для скачивания и анализа выполните следующую ячейку


In [7]:
# ============================================================
# STAGE 7.5 — Запуск: вставьте свою ссылку
# ============================================================

# ← Вставьте сюда любую YouTube ссылку
YOUTUBE_URL = 'https://www.youtube.com/shorts/5kNIa6-5eHI'

# Шаг 1: скачиваем
video_path = download_youtube(YOUTUBE_URL)

# Шаг 2: предсказываем
if video_path:
    results = predict_genre(video_path, top_k=5)
    print(f'\n🏆 Итог: {GENRE_EMOJI.get(results[0][0], "")} {results[0][0]} ({results[0][1]*100:.1f}%)')


📥 Скачиваем: https://www.youtube.com/shorts/5kNIa6-5eHI
✅ Скачано: 5kNIa6-5eHI.mp4  (2.0 MB)

🎬 Анализируем: 5kNIa6-5eHI.mp4
   Извлекаем фичи... 

I0000 00:00:1774288144.521920  162059 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
I0000 00:00:1774288145.504319  162181 cuda_dnn.cc:461] Loaded cuDNN version 91002
W0000 00:00:1774288145.664150  162181 nvptx_libdevice_path.cc:41] Can't find libdevice directory ${CUDA_DIR}/nvvm/libdevice. This may result in compilation or runtime failures, if the program we try to run uses routines from libdevice.
Searched for CUDA in the following directories:
  ./cuda_sdk_lib
  ipykernel_launcher.runfiles/cuda_nvcc
  ipykernel_launcher.runfiles/cuda_nvdisasm
  ipykernel_launcher.runfiles/nvidia_nvshmem
  ipykernel_launcher.runfiles/cuda_nvvm
  ipykernel_launcher.runfiles/cuda_cudart
  /usr/local/cuda
  /opt/cuda
  /home/wukker/IdeaProjects/video_classifier/.venv/lib/python3.12/site-packages/tensorflow/python/platform/../../../nvidia/cuda_nvcc
  /home/wukker/IdeaProjects/video_classifier/.venv/lib/python3.12/site-packages/tensorflow/python/platform/../../../../nvidia/c

✅
   RGB shape=(1024,)  Audio shape=(128,)

   📊 Результаты:
   1. 🎨 Animation        44.6%  █████████████
   2. 🎮 Gaming           43.1%  ████████████
   3. 🎬 Film              5.2%  █
   4. 🎵 Music             4.1%  █
   5. 🐾 Animals           0.6%  

🏆 Итог: 🎨 Animation (44.6%)


In [8]:
# ============================================================
# STAGE 7.6 — Batch: несколько ссылок сразу
# ============================================================

test_urls = [
    'https://www.youtube.com/shorts/5kNIa6-5eHI', # minecraft animation
    'https://youtu.be/dQw4w9WgXcQ?si=0Blx38yz_nMFEB1O',  # rick-roll
    'https://youtube.com/shorts/7PVrmGOim90?si=oYdVExRrfOCns2T4',  # muscle training guide
    'https://youtube.com/shorts/-KnFzkY9Kro?si=0kjThdi3KRKgyCap',  # dance clip
]

print('=' * 55)
print('   BATCH INFERENCE')
print('=' * 55)
for url in test_urls:
    vid = download_youtube(url)
    if vid:
        res = predict_genre(vid, top_k=3, verbose=True)
    print()


   BATCH INFERENCE
📥 Скачиваем: https://www.youtube.com/shorts/5kNIa6-5eHI
✅ Скачано: 5kNIa6-5eHI.mp4  (2.0 MB)

🎬 Анализируем: 5kNIa6-5eHI.mp4
   Извлекаем фичи... ✅
   RGB shape=(1024,)  Audio shape=(128,)

   📊 Результаты:
   1. 🎨 Animation        44.6%  █████████████
   2. 🎮 Gaming           43.1%  ████████████
   3. 🎬 Film              5.2%  █

📥 Скачиваем: https://youtu.be/dQw4w9WgXcQ?si=0Blx38yz_nMFEB1O
✅ Скачано: dQw4w9WgXcQ.mp4  (28.5 MB)

🎬 Анализируем: dQw4w9WgXcQ.mp4
   Извлекаем фичи... ✅
   RGB shape=(1024,)  Audio shape=(128,)

   📊 Результаты:
   1. 🎬 Film             47.1%  ██████████████
   2. 🎵 Music            19.0%  █████
   3. 🎭 Performance      12.1%  ███

📥 Скачиваем: https://youtube.com/shorts/7PVrmGOim90?si=oYdVExRrfOCns2T4
✅ Скачано: 7PVrmGOim90.mp4  (0.6 MB)

🎬 Анализируем: 7PVrmGOim90.mp4
   Извлекаем фичи... ✅
   RGB shape=(1024,)  Audio shape=(128,)

   📊 Результаты:
   1. 🎵 Music            48.6%  ██████████████
   2. 💃 Dance            19.5%  █████
   3